# Natural Language Processing - Assignment 1

### Group 35

Members:


*   Mehdi Hellal
*   Antonio Augusto Brito de Sousa
*   Adam Oprchal



## Introduction

In this assignment, we train various NLP classifiers on [The Biggest Spam Ham Phish Email Dataset (250000+)](https://www.kaggle.com/datasets/akshatsharma2/the-biggest-spam-ham-phish-email-dataset-300000). This dataset contains over 250000 English messages and emails annotated into three classes: spam (unwanted mass message/email), ham (safe, normal message/email) and phish (malicious type of spam designed to steal sensitive information).

This dataset was created by merging other publicly available and open-source datasets and properly cleaned and deduplicated. Labeling of the data was also verified. The labels in the dataset have this mapping: 0 → Ham, 1 → Phish, 2 → Spam.

## Exploratory Data Analysis

Let's download the dataset and load it into pandas.

In [ ]:
import kagglehub

path = kagglehub.dataset_download("akshatsharma2/the-biggest-spam-ham-phish-email-dataset-300000")

In [ ]:
import pandas as pd

dataset = pd.read_csv(path + "/df.csv")
dataset.head()

In [ ]:
dataset.shape

In [ ]:
dataset["text"].drop_duplicates().shape

In [ ]:
dataset.drop_duplicates().shape

In [ ]:
dataset.dropna().shape

We can see that the dataset contain 2 columns: the label and the message. The dataset has over 360000 messages, but only about 280000 unique ones. This is in accordance with the dataset documentation, but still somewhat confusing, because the dataset should be deduplicated. 8 of the duplicated messages have different labels, but this might be because of some ambiguity in the messages. The dataset also contains some rows with missing values.

Let's remove only the rows which are total duplicates and also the rows with missing values.

In [ ]:
dataset = dataset.dropna().drop_duplicates()
dataset.shape

Let's add a new label column for better clarity and explore the dataset even more.

In [ ]:
mapping = {0: "ham", 1: "phish", 2: "spam"}

dataset["named_label"] = dataset["label"].map(mapping)

In [ ]:
dataset["named_label"].value_counts()

In [ ]:
dataset["named_label"].value_counts().plot(kind="bar")

We can see that the dataset contains similar amount of ham and spam messages, but not a lot of phish. This means that we have mildly imbalanced classes and could encounter some problems in our assignment.

The bigger issue is the size of the dataset. We decided to only take a fraction of all rows to speed up the preprocessing and training process, because otherwise it would take a very long time. We also use stratified sampling to solve the issue with imbalanced classes.

In [ ]:
min_count = dataset["label"].value_counts().min()
n_samples = int(min_count * 0.1)

dataset = (
    dataset.groupby("label")
      .apply(lambda x: x.sample(n=n_samples, random_state=42))
      .reset_index(drop=True)
)

In [ ]:
dataset["named_label"].value_counts().plot(kind="bar")

Now we can see that the dataset looks nice and compact.

We can now observe the lengths of the messages, both in words and total characters, by each class.

In [ ]:
dataset["text_character_count"] = dataset["text"].str.len()
dataset["text_word_count"] = dataset["text"].str.split().str.len()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(x="named_label", y="text_word_count", data=dataset)
plt.show()

We can see that there are quite a lot of outliers when it comes to the number of words of each message. Let's hide the outliers.

In [ ]:
sns.boxplot(x="named_label", y="text_word_count", data=dataset, showfliers=False)
plt.show()

And now the character count in each message (outliers were also hidden for the same reason).

In [ ]:
sns.boxplot(x="named_label", y="text_character_count", data=dataset, showfliers=False)
plt.show()

We can conclude that the lengths of the messages across the classes are roughly the same. Phish has slightly shorter texts, but nothing too extremely significant.

Let's explore the texts of the messages.

In [ ]:
dataset["text"][0]

In [ ]:
dataset["text"][1]

In [ ]:
dataset["text"][2]

We can see that the texts are already slightly preprocessed. It looks like the messages were lowercased and some numbers were replaced by escapenumber. The texts might also contain '\n' to indicate a new paragraph.

Before any additional preprocessing, let's look at the most common words in the texts. We can use N-grams for this. - ehh maybe? they will not tell us the probability of some pairs of words, only the probability of a word following some previous context, which might not be that useful...... consider this

let's remove noise : "escapenumber" and "escapelong" and variants

In [ ]:
import re

def clean_text(text):
    text = text.lower()

    # remove any word containing "escapenumber" or "escapelong"
    text = re.sub(r'\b\w*escape(num(ber|b)?|long)\w*\b', '', text)

    # remove non-alphabetic characters
    text = re.sub(r'[^a-z\s]', '', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def get_top_ngrams(texts, ngram_range=(1,2), top_n=15):
    vec = CountVectorizer(ngram_range=ngram_range, stop_words="english")
    X = vec.fit_transform(texts)
    counts = X.sum(axis=0).A1
    terms = vec.get_feature_names_out()

    top_idx = counts.argsort()[::-1][:top_n]
    return pd.DataFrame({
        "term": terms[top_idx],
        "count": counts[top_idx]
    })


for label in ["ham", "phish", "spam"]:
    texts = dataset[dataset["named_label"] == label]["text"].apply(clean_text)

    top_bigrams = get_top_ngrams(
        texts,
        ngram_range=(2,3)
    )

    plt.figure(figsize=(10, 5))
    sns.barplot(data=top_bigrams, x="count", y="term")
    plt.title(f"Top 15 bigram + trigrams in {label} emails")
    plt.xlabel("Frequency")
    plt.ylabel("ngram")
    plt.show()

## Preprocessing

Now for the preprocessing. Let's remove any non-alphabetic characters and English language stop words and stem the remaining words. Let's also lowercase the texts just in case.

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from tqdm import tqdm

ps = PorterStemmer()

nltk.download('stopwords')
sw = set(stopwords.words("english"))

preprocessed_texts = []
for i in tqdm(range(0, dataset["text"].size)):
    # get message text and remove non alpha chars
    text = re.sub("[^a-zA-Z]", " ", dataset["text"][i])
    # to lower-case
    text = text.lower()
    # split into tokens, apply stemming and remove stop words
    text = " ".join([ps.stem(w) for w in text.split() if w not in sw])
    preprocessed_texts.append(text)

dataset["preprocessed_text"] = preprocessed_texts

In [ ]:
print("Total texts length before preprocess: ", dataset["text"].str.len().sum())
print("Total texts length after preprocess: ", dataset["preprocessed_text"].str.len().sum())
print("Texts reduced to: ", "{:.2f}".format(dataset["preprocessed_text"].str.len().sum() / dataset["text"].str.len().sum() * 100), "% of original length")

In [ ]:
dataset["preprocessed_text"][0]

We can now look at word clouds of the preprocessed texts.

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wordcloud = WordCloud().generate(" ".join(dataset["preprocessed_text"]))

plt.figure()
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

In [ ]:
wordcloud = WordCloud().generate(" ".join(dataset[dataset["named_label"] == "spam"]["preprocessed_text"]))

plt.figure()
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

In [ ]:
wordcloud = WordCloud().generate(" ".join(dataset[dataset["named_label"] == "ham"]["preprocessed_text"]))

plt.figure()
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

In [ ]:
wordcloud = WordCloud().generate(" ".join(dataset[dataset["named_label"] == "phish"]["preprocessed_text"]))

plt.figure()
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

They look quite unclear, look at something like TF-IDF for better clarity.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def get_top_tfidf_terms(texts, top_n=15):
    # apply cleaning
    texts = texts.apply(clean_text)

    vec = TfidfVectorizer(stop_words="english", max_features=5000)
    X = vec.fit_transform(texts)

    mean_tfidf = np.asarray(X.mean(axis=0)).ravel()
    terms = vec.get_feature_names_out()

    top_idx = mean_tfidf.argsort()[::-1][:top_n]

    return pd.DataFrame({
        "term": terms[top_idx],
        "mean_tfidf": mean_tfidf[top_idx]
    })

In [ ]:
for label in ["ham", "phish", "spam"]:
    top_terms = get_top_tfidf_terms(
        dataset[dataset["named_label"] == label]["preprocessed_text"],
        top_n=10
    )

    plt.figure(figsize=(10, 5))
    sns.barplot(data=top_terms, x="mean_tfidf", y="term")
    plt.title(f"Top 10 TF-IDF terms in {label} emails")
    plt.xlabel("Average TF-IDF")
    plt.ylabel("Term")
    plt.show()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

for label in ["ham", "phish", "spam"]:
    texts = dataset[dataset["named_label"] == label]["preprocessed_text"]
    vec = CountVectorizer()
    vec.fit(texts)
    print(f"{label.upper()} vocabulary size: {len(vec.vocabulary_)}")

In [ ]:
def lexical_richness(text):
    words = text.split()
    if len(words) == 0:
        return 0
    return len(set(words)) / len(words)

dataset["lexical_richness"] = dataset["preprocessed_text"].apply(lexical_richness)

plt.figure(figsize=(8,5))
sns.boxplot(x="named_label", y="lexical_richness", data=dataset, showfliers=False)
plt.title("Lexical richness by class")
plt.xlabel("Class")
plt.ylabel("Unique words / total words")
plt.show()

## Baseline Models : Bag-of-words + ( NB / LogisticRegression / MLP )

### Bag-of-words Feature Representation



To start, we are going to use the simplest feature representation technique - bag-of-words with 1000 features.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features = 1000)
X = vectorizer.fit_transform(dataset["preprocessed_text"]).toarray()
y = dataset["label"]

print(X.shape)

### Bag-of-words Classification

#### Train Test Split & evaluation function

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, stratify=y)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

print("\nLabel distribution in the training set:")
print(y_train.value_counts())

print("\nLabel distribution in the test set:")
print(y_test.value_counts())

A helper function for our models:

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score

def evaluate_model(clf, X_test, y_test, normalize=False):
    y_pred = clf.predict(X_test)

    # Compute confusion matrix with fixed label order
    labels = [0, 1, 2]
    class_names = ["Ham", "Phish", "Spam"]
    cm = confusion_matrix(y_test, y_pred, labels=labels)

    # Normalize if requested
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    # Plot heatmap
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm,
                annot=True,
                fmt='.2f' if normalize else 'd',
                cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names)

    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix" + (" (Normalized)" if normalize else ""))
    plt.show()

    # Accuracy
    acc = accuracy_score(y_test, y_pred)
    print("Accuracy:", "{:.2f}".format(acc * 100), "%")

#### BoW + Naive bayes

Let's start with Naive Bayes model as our baseline.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

clf = MultinomialNB()
clf.fit(X_train, y_train)

evaluate_model(clf, X_test, y_test)

#### BoW + LogisticRegression

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()
clf.fit(X_train, y_train)

evaluate_model(clf, X_test, y_test)

#### BoW + MLP

Let's try something overkill.

In [ ]:
from sklearn.neural_network import MLPClassifier

clf = MLPClassifier(hidden_layer_sizes=(500, 500), early_stopping=True, max_iter=5000)
clf.fit(X_train, y_train)

evaluate_model(clf, X_test, y_test)

Not that great, we should try at least one more feature representation, most likely a dense representation.

### Hyperparameter tuning with GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

#### Bag-of-Words + Logistic Regression tuning

In [ ]:
param_grid_bow_lr = {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["liblinear", "lbfgs"],
    "max_iter": [1000, 2000]
}

grid_bow_lr = GridSearchCV(
    estimator=LogisticRegression(),
    param_grid=param_grid_bow_lr,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid_bow_lr.fit(X_train, y_train)

print("Best parameters for BoW + Logistic Regression:")
print(grid_bow_lr.best_params_)
print()
print("Best cross-validation accuracy: {:.4f}".format(grid_bow_lr.best_score_))

In [ ]:
best_bow_lr = grid_bow_lr.best_estimator_
evaluate_model(best_bow_lr, X_test, y_test)

## Dense representation models : Word2Vec + ( LogisticRegression / RandomForest / MLP )

### Word2Vec Feature Representation

In [ ]:
!pip install gensim

In [ ]:
import gensim

dataset["gensim_preprocess"] = dataset["text"].apply(gensim.utils.simple_preprocess)

We train embeddings only on the train dataset.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(dataset, dataset["label"], test_size = 0.20, stratify=y)

In [ ]:
from datetime import datetime

start_time = datetime.now()

model = gensim.models.Word2Vec(X_train["gensim_preprocess"], vector_size=150, window=10, min_count=2, workers=10, sg=1)

print("Training time:", datetime.now() - start_time)

For each text, we get its vector by taking the mean of the vectors of individual words.

In [ ]:
def text_to_mean_vector(text):
    vectors = []

    for token in text:
        if token in model.wv:
            vectors.append(model.wv[token])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [ ]:
X_train = X_train["gensim_preprocess"].apply(text_to_mean_vector)
X_test = X_test["gensim_preprocess"].apply(text_to_mean_vector)

X_train = np.vstack(X_train.values)
X_test = np.vstack(X_test.values)

In [ ]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

print("\nLabel distribution in the training set:")
print(y_train.value_counts())

print("\nLabel distribution in the test set:")
print(y_test.value_counts())

### Word2Vec Classification

#### Word2Vec + LogisticRegression

In [ ]:
clf = LogisticRegression()
clf.fit(X_train, y_train)

evaluate_model(clf, X_test, y_test)

#### Word2Vec + RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier()
clf.fit(X_train, y_train)

evaluate_model(clf, X_test, y_test)

#### Word2Vec + MLP

##### 2 hidden layers

In [ ]:
from sklearn.neural_network import MLPClassifier

clf = MLPClassifier(hidden_layer_sizes=(500, 500), early_stopping=True, max_iter=5000)
clf.fit(X_train, y_train)

evaluate_model(clf, X_test, y_test)

##### 3 hidden layers

In [ ]:
from sklearn.neural_network import MLPClassifier

clf = MLPClassifier(hidden_layer_sizes=(500, 500, 500), early_stopping=True, max_iter=5000)
clf.fit(X_train, y_train)

evaluate_model(clf, X_test, y_test)

### Hyperparameter tuning with GridSearchCV

#### Word2Vec + MLP tuning

In [ ]:
param_grid_w2v_mlp = {
    "hidden_layer_sizes": [
        (200, 200, 200),
        (300, 300, 300),
        (500, 500, 500)
    ],
    "activation": ["relu"],
    "alpha": [0.0001, 0.001],
    "learning_rate_init": [0.001],
    "max_iter": [5000],
    "early_stopping": [True]
}

grid_w2v_mlp = GridSearchCV(
    estimator=MLPClassifier(),
    param_grid=param_grid_w2v_mlp,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid_w2v_mlp.fit(X_train, y_train)

print("Best parameters for Word2Vec + MLP:")
print(grid_w2v_mlp.best_params_)
print()
print("Best cross-validation accuracy: {:.4f}".format(grid_w2v_mlp.best_score_))

In [ ]:
best_w2v_mlp = grid_w2v_mlp.best_estimator_

evaluate_model(best_w2v_mlp, X_test, y_test)


# Error Analysis

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

# 1. Extração das previsões baseada no teu fluxo de treino
y_pred_nb = clf.predict(X_test)
y_pred_lr = grid_bow_lr.best_estimator_.predict(X_test)
y_pred_mlp = clf.predict(X_test)


acc_nb = accuracy_score(y_test, y_pred_nb)
acc_lr = accuracy_score(y_test, y_pred_lr)
acc_mlp = accuracy_score(y_test, y_pred_mlp)

accuracies = [acc_nb * 100, acc_lr * 100, acc_mlp * 100]
models = ['Naive Bayes', 'LogReg (Grid)', 'MLP']


plt.figure(figsize=(8, 5))
sns.barplot(x=models, y=accuracies, palette='viridis')
plt.ylim(min(accuracies) - 2, 100)
plt.title('Final Accuracy Comparison')
plt.ylabel('Accuracy (%)')
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.5, f"{v:.2f}%", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


error_df = pd.DataFrame({
    'Text': dataset.loc[y_test.index, 'text'],
    'Actual': y_test.values,
    'Predicted': y_pred_lr
})
misclassified = error_df[error_df['Actual'] != error_df['Predicted']]

plt.figure(figsize=(10, 6))
error_counts = misclassified.groupby(['Actual', 'Predicted']).size().reset_index(name='Total')
error_counts['Label'] = error_counts.apply(lambda x: f"Actual {int(x['Actual'])} vs Predicted {int(x['Predicted'])}", axis=1)

sns.barplot(data=error_counts, x='Label', y='Total', color='salmon')

plt.xticks(rotation=45, ha='right')
plt.title(f'Analysis of Misclassifications (Total: {len(misclassified)} errors)')
plt.tight_layout()
plt.show()

1. Quantitative Performance
The three models performed very similarly, with Logistic Regression (GridSearch) slightly leading at 81.32%, followed by Naive Bayes and MLP, both at 80.93%.

2. Misclassification Analysis (48 Total Errors)
With 48 errors, we can observe a clear pattern in the model's failure points:

Feature Sparsity in Short Emails: Many errors occur in very short messages.

Semantic Ambiguity: The confusion between Spam (1) and Phishing (2) remains the primary source of error. Both classes frequently use words like "account," "login," "verify," and "urgent." Without sequential context (which BoW ignores), the models struggle to distinguish a marketing "call to action" from a malicious "lure."

In [ ]:
import pandas as pd


best_dense_model = grid_w2v_mlp.best_estimator_
y_pred_dense = best_dense_model.predict(X_test)


error_df_dense = pd.DataFrame({
    'Text': dataset.loc[y_test.index, 'text'],
    'Actual': y_test.values,
    'Predicted': y_pred_dense
})

misclassified_dense = error_df_dense[error_df_dense['Actual'] != error_df_dense['Predicted']]

print(f"Total Errors (Dense): {len(misclassified_dense)}")
print("\n--- Dense Model Error Samples ---")
print(misclassified_dense[['Text', 'Actual', 'Predicted']].head(5))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


plt.figure(figsize=(10, 6))
error_counts_dense = misclassified_dense.groupby(['Actual', 'Predicted']).size().reset_index(name='Total')
error_counts_dense['Label'] = error_counts_dense.apply(lambda x: f"Actual {int(x['Actual'])} vs Pred {int(x['Predicted'])}", axis=1)

sns.barplot(data=error_counts_dense, x='Label', y='Total', palette='viridis')
plt.xticks(rotation=45, ha='right')
plt.title(f'Dense Model Error Analysis (Total: {len(misclassified_dense)} errors)')
plt.ylabel('Number of Errors')
plt.tight_layout()
plt.show()


final_comparison = pd.DataFrame({
    'Approach': ['Sparse (BoW)', 'Dense (Word2Vec)'],
    'Errors': [48, 26]
})

plt.figure(figsize=(6, 5))
sns.barplot(data=final_comparison, x='Approach', y='Errors', palette='coolwarm')
plt.title('Total Error Comparison: Sparse vs Dense')
plt.ylabel('Total Errors (Lower is Better)')
for i, v in enumerate(final_comparison['Errors']):
    plt.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
plt.show()


 Moving from a Sparse (Bag-of-Words) approach to a Dense (Word2Vec) representation resulted in a substantial decrease in total errors, dropping from 48 to 26. This represents a 45.8% improvement in classification accuracy.

Unlike Sparse models that treat words as isolated tokens, the Dense model captures latent semantic relationships. This suggests that understanding the context of words (e.g., the relationship between "urgent," "verify," and "account") is crucial for effectively distinguishing between Ham, Phish, and Spam.
The Dense approach achieved superior results using only 150 dimensions compared to the 1,000 dimensions used in the Sparse model. This proves that dense embeddings are a more compact and expressive way to represent textual information for neural networks.